In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch

random.seed(24)
np.random.seed(24)

In [ ]:
device = torch.device("cuda")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(24)
torch.cuda.manual_seed_all(24)

---
## Markov Decision Process (MDP) Framework

In [ ]:
class GridWorld:
    """
    A simple 4x4 GridWorld MDP.
    State: (row, col). Goal: reach (3,3). Pit: (2,2).
    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    """
    def __init__(self, size=4):
        self.size = size
        self.n_states = size * size
        self.n_actions = 4
        self.goal = (size-1, size-1)
        self.pit = (size//2, size//2)
        self.action_effects = [(-1,0),(1,0),(0,-1),(0,1)]  # U D L R

    def state_id(self, s): return s[0]*self.size + s[1]
    def state_xy(self, sid): return (sid // self.size, sid % self.size)

    def reset(self):
        self.pos = (0, 0)
        return self.state_id(self.pos)

    def step(self, action):
        dr, dc = self.action_effects[action]
        r, c = self.pos
        nr = max(0, min(self.size-1, r + dr))
        nc = max(0, min(self.size-1, c + dc))
        self.pos = (nr, nc)
        done = False
        if self.pos == self.goal:
            reward, done = +10.0, True
        elif self.pos == self.pit:
            reward, done = -10.0, True
        else:
            reward = -0.1  # small step penalty encourages efficiency
        return self.state_id(self.pos), reward, done

env = GridWorld()
print(f"GridWorld: {env.n_states} states, {env.n_actions} actions")
print(f"Goal state id: {env.state_id(env.goal)}, Pit state id: {env.state_id(env.pit)}")

## Softmax Policy

In [ ]:
class SoftmaxPolicy:
    """
    Tabular softmax policy: pi(a|s; theta) = exp(theta[s,a]) / sum_a' exp(theta[s,a'])
    This parametrizes pi differentiably, as required by the policy gradient theorem.
    """
    def __init__(self, n_states, n_actions, temperature=1.0, device=device):
        self.theta = torch.zeros((n_states, n_actions), dtype=torch.float32, device=device)
        self.tau = temperature
        self.device = device
        self.n_actions = n_actions

    def probs(self, s):
        logits = self.theta[s] / self.tau
        return torch.softmax(logits, dim=0)

    def select_action(self, s):
        probs = self.probs(s)
        return torch.multinomial(probs, num_samples=1).item()

    def log_grad(self, s, a):
        """Gradient of log pi(a|s) w.r.t. theta[s,:] (one-hot minus probs)."""
        p = self.probs(s)
        grad = -p.clone()
        grad[a] += 1.0
        return grad

---
## Actor-Critic

The Actor-Critic method maintains two networks:
- **Critic**: estimates $Q(s,a;w) = \phi(s,a)^\top w$ using TD(0)
- **Actor**: updates policy parameter $\theta$ using the critic's estimate

$$w^{(n+1)} \leftarrow w^{(n)} + \beta_w \, \delta \, \phi(s,a)$$
$$\theta^{(n+1)} \leftarrow \theta^{(n)} + \beta_\theta \, Q(s,a;w^{(n)}) \, \nabla_\theta \ln \pi(s,a;\theta^{(n)})$$

In [ ]:
def actor_critic(env, num_episodes=5000, gamma=0.99, lr_actor=0.01, lr_critic=0.05, device=device):
    """
    Algorithm 5 (paper): Actor-Critic.
    Critic: linear function approximation Q(s,a;w) = phi(s,a)^T w
    Actor: softmax policy parametrized by theta
    Single-scale setting: actor and critic update simultaneously.
    """
    n_s, n_a = env.n_states, env.n_actions
    n_features = n_s * n_a

    def feature_index(s, a):
        return s * n_a + a

    w = torch.zeros(n_features, dtype=torch.float32, device=device)
    policy = SoftmaxPolicy(n_s, n_a, device=device)
    cum_rewards = []

    for ep in range(num_episodes):
        s = env.reset()
        a = policy.select_action(s)
        done = False
        total_r = 0.0

        while not done:
            s_next, r, done = env.step(a)
            total_r += r
            reward_t = torch.tensor(r, dtype=torch.float32, device=device)
            a_next = policy.select_action(s_next)

            current_idx = feature_index(s, a)
            next_idx = feature_index(s_next, a_next)

            q_sa = w[current_idx]
            q_next = w[next_idx] if not done else torch.tensor(0.0, dtype=torch.float32, device=device)
            delta = reward_t + gamma * q_next - q_sa

            w[current_idx] += lr_critic * delta
            policy.theta[s] += lr_actor * q_sa.detach() * policy.log_grad(s, a)

            s, a = s_next, a_next

        cum_rewards.append(total_r)

    return policy, w, cum_rewards


policy_ac, w_ac, rewards_ac = actor_critic(env, num_episodes=5000, device=device)
print(f"Policy parameters live on: {policy_ac.theta.device}")
print(f"Critic parameters live on: {w_ac.device}")

window = 100
reward_tensor = torch.tensor(rewards_ac, dtype=torch.float32, device=device)
kernel = torch.ones(window, dtype=torch.float32, device=device) / window
smooth = torch.nn.functional.conv1d(
    reward_tensor.view(1, 1, -1),
    kernel.view(1, 1, -1)
).view(-1)
smooth_cpu = smooth.detach().cpu().numpy()

plt.figure(figsize=(12, 5))
plt.plot(smooth_cpu, label='Actor-Critic', linewidth=1.5, color='orchid')
plt.xlabel('Episode')
plt.ylabel('Cumulative Reward (100-ep smoothed)')
plt.title(f'Actor-Critic on GridWorld ({device.type.upper()})')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final smoothed reward: {smooth[-1].item():.3f}")
